In [1]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
data_path = "/content/drive/MyDrive/big_data_project"

In [4]:
from rasterio.plot import show

import matplotlib.pyplot as plt

tif_path = r"Data/soil/HWSD2_RASTER/HWSD2.bil"
tif_path = os.path.join(data_path, tif_path)


In [5]:
!ls -l /content/drive/MyDrive/big_data_project/Data/soil/HWSD2_RASTER/

total 1822502
-rw------- 1 root root 1866240000 Nov 11  2022 HWSD2.bil
-rw------- 1 root root        338 Nov 11  2022 HWSD2.hdr
-rw------- 1 root root        146 Nov 11  2022 HWSD2.prj
-rw------- 1 root root         64 Nov 11  2022 HWSD2.stx


In [6]:
import os
size_gb = os.path.getsize(tif_path) / (1024**3)
print(f"Taille du fichier : {size_gb:.2f} GB")

Taille du fichier : 1.74 GB


In [7]:
with rasterio.open(tif_path) as ds:
    print(ds.count)

1


In [8]:
def preparer_donnees_soil():
   
    
    df_grille = pd.read_csv("/content/drive/MyDrive/big_data_project/Data/csv_final/grid_portugal_1km.csv")
    

    print(f"Grille: {len(df_grille)} points")
    
    
    df_complet = df_grille[['longitude', 'latitude']].copy()
    tif_path = os.path.join(data_path, r"Data/soil/HWSD2_RASTER/HWSD2.bil")   
    valeurs = extraction_vectorisee(tif_path, df_grille)
    df_complet["SMU_ID"] = valeurs
            
    valides = np.sum(~np.isnan(valeurs))
    print(f"({valides} valides)")
    
    # Sauvegarde
    output_csv = os.path.join(data_path, "Data/soil/donnees_soil_collab.csv")
    df_complet.to_csv(output_csv, index=False)
    
    return df_complet

# Version vectorisée à implémenter
def extraction_vectorisee(tif_path, df_points):
    with rasterio.open(tif_path) as src:
        # Conversion de TOUTES les coordonnées en une fois
        rows, cols = rasterio.transform.rowcol(
            src.transform, 
            df_points['longitude'].values, 
            df_points['latitude'].values
        )
        
        # Masque des points valides
        mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        
        # Initialisation du tableau de résultats
        valeurs = np.full(len(df_points), np.nan)
        
        if np.any(mask):
            # Extraction des valeurs valides
            data = src.read(1)
            valeurs_valides = data[rows[mask], cols[mask]]
            
            # Application du masque
            valeurs[mask] = valeurs_valides
            
            # Gestion NoData
            valeurs[valeurs == src.nodata] = np.nan
    
    return valeurs

# Essayez cette version plus rapide
data = preparer_donnees_soil()

Grille: 89198 points
(89112 valides)


In [9]:
data.isna().sum()

,0
longitude,0
latitude,0
SMU_ID,86


In [10]:
data[data["SMU_ID"].isna()].head(86)

,longitude,latitude,SMU_ID
3,-8.952309,37.003139,NaN
9,-7.929930,37.005792,NaN
81,-8.885261,37.039625,NaN
1170,-7.546833,37.148687,NaN
1418,-8.909117,37.174628,NaN
...,...,...,...
69757,-8.655396,41.040236,NaN
71018,-8.667872,41.112214,NaN
71987,-8.692141,41.166125,NaN
73689,-8.728702,41.255984,NaN


In [11]:
from scipy.spatial import cKDTree

# Séparer les points valides et NaN
valid_mask = data["SMU_ID"].notna()
nan_mask = ~valid_mask

valid_coords = data.loc[valid_mask, ["longitude", "latitude"]].values
nan_coords = data.loc[nan_mask, ["longitude", "latitude"]].values
valid_values = data.loc[valid_mask, "SMU_ID"].values

# Construire l'arbre KD et trouver le plus proche voisin
tree = cKDTree(valid_coords)
_, idx = tree.query(nan_coords, k=1)

# Remplacer les NaN par la valeur la plus proche
data.loc[nan_mask, "SMU_ID"] = valid_values[idx]

In [12]:
data.isna().sum()

,0
longitude,0
latitude,0
SMU_ID,0


In [13]:
output_csv = "/content/drive/MyDrive/big_data_project/Data/soil/donnees_soil_1km_without_missing_valuesCollab.csv"
data.to_csv(output_csv, index=False)

In [14]:
!pip install pyodbc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 6.8 MB/s eta 0:00:00a 0:00:01


In [15]:
from rasterio.enums import Resampling
import requests
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import mapping
import pyodbc